In [ ]:
!pip uninstall langchain langchain-core langchain-sarvam langchain-classic langchain-text-splitters -y

In [ ]:
!pip install langchain langchain-core langchain-sarvam langchain-classic langchain-text-splitters 

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_sarvam import ChatSarvam
from kaggle_secrets import UserSecretsClient
import os 

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [ ]:
my_prompt_template = """
      ## Your task is to explain the meaning of the  error. The error name is **{Error_name}** that comes in **{Role}** in the way that is :

          1. Return the output clearly and intuitively
          2. Use Proper Formating
          3. What is the meaning of the error
          4. When it occurs
          5. How can I fix it
          6. Use an analogy ot explaning the error, and how can I avoid the same mistake next time

      ## Use the following information about me to personalize your explanations:

      - Background: Data engineering student, also studying AI engineering. Experienced with PySpark, Pandas, and advanced SQL. Has built projects with AI tools.
      - Professional Interests: Building LLM applications for data engineering, such as ad hoc question answering and data documentation, and normalizing nested data.
      - Personal Interests: AI/ML, model fine-tuning and evaluation.

      The personalization should be subtle and natural. Avoid forced references to my background or interests that don't genuinely enhance my understanding of the concept.

      ## The exact **Error** that is returned during code execution is given below :

      {Error_message}
    """

In [ ]:
prompt_template = PromptTemplate.from_template(my_prompt_template)
ErrorName    = "ModuleNotFoundError and ImportError"
CodeRole     = "Langchain"

In [ ]:
ErrorMessage = """
 ---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
/usr/local/lib/python3.13/dist-packages/langchain/chat_models/base.py in _import_module(module, class_name)
    139     try:
--> 140         return importlib.import_module(module)
    141     except ImportError as e:

ModuleNotFoundError: No module named 'langchain_openai'

ImportError: Initializing ChatOpenAI requires the langchain-openai package.
Please install it with `pip install langchain-openai`
"""

In [ ]:
secret_sarvam_key = user_secrets.get_secret("SARVAM_AI_API")
if not secret_sarvam_key:
    raise RuntimeError("Set SERVAM_AI_API in your Colab Secrets before running.")

# Set the environment variable that ChatSarvam expects
os.environ["SARVAM_API_KEY"] = secret_sarvam_key

In [ ]:
llm = ChatSarvam(
    model="sarvam-105b",
    sarvam_api_key=secret_sarvam_key,
    max_retries=1
)

In [ ]:
def generate_explanation(error_name: str, role: str, error_message: str):
  user_input = {
      "Error_name": error_name,
      "Role": role,
      "Error_message": error_message
  }
  formatted_template = prompt_template.invoke(user_input)
  response = llm.invoke(formatted_template)
  return response.content

In [ ]:
from IPython.display import display, Markdown

# Run the generator and format the markdown cleanly
explanation_md = generate_explanation(ErrorName, CodeRole, ErrorMessage)
display(Markdown(explanation_md))